In [0]:
from pyspark.sql.functions import (
    col, sum as spark_sum, count, avg, round as spark_round,
    countDistinct, when, lit, rank, current_timestamp
)
from pyspark.sql.window import Window

print("Building Gold layer...")

# ---- DIMENSION TABLE: dim_customers ----
df_dim_customers = spark.table("wholesale_catalog.silver.customers") \
    .select("customer_id", "customer_name", "region", "country", "email") \
    .withColumn("gold_updated_at", current_timestamp())

df_dim_customers.write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("wholesale_catalog.gold.dim_customers")

print("dim_customers done")

# ---- DIMENSION TABLE: dim_products ----
df_dim_products = spark.table("wholesale_catalog.silver.products") \
    .select("product_id", "product_name", "category", "unit_price") \
    .withColumn("gold_updated_at", current_timestamp())

df_dim_products.write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("wholesale_catalog.gold.dim_products")

print("dim_products done")


In [0]:
# ---- FACT TABLE: fact_sales ----
# Join line items + invoices + exchange rates for USD-converted revenue

df_lines    = spark.table("wholesale_catalog.silver.invoice_line_items")
df_invoices = spark.table("wholesale_catalog.silver.invoices")
df_fx       = spark.table("wholesale_catalog.silver.exchange_rates")

# Get latest exchange rate per currency
from pyspark.sql.functions import max as spark_max
df_fx_latest = df_fx.groupBy("currency") \
    .agg(spark_max("rate_to_usd").alias("rate_to_usd"))

df_fact_sales = df_lines \
    .join(df_invoices, "invoice_id", "left") \
    .join(df_fx_latest, df_invoices["currency"] == df_fx_latest["currency"], "left") \
    .withColumn("line_total_usd", col("line_total") * coalesce(col("rate_to_usd"), lit(1.0))) \
    .select(
        "line_item_id", "invoice_id", "product_id",
        df_invoices["customer_id"],
        df_invoices["invoice_date"],
        df_invoices["invoice_status"],
        df_invoices["currency"],
        "quantity", "unit_price", "discount_pct",
        "line_total", "line_total_usd",
        df_invoices["region"] if "region" in df_invoices.columns else lit(None).alias("region")
    ) \
    .withColumn("gold_updated_at", current_timestamp())

df_fact_sales.write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("wholesale_catalog.gold.fact_sales")

print(f"fact_sales: {df_fact_sales.count()} rows")

# ============================================================
# KPI 1: Total Revenue (USD)
# ============================================================
df_kpi_total_revenue = spark.table("wholesale_catalog.gold.fact_sales") \
    .filter(col("invoice_status") != "CANCELLED") \
    .agg(spark_round(spark_sum("line_total_usd"), 2).alias("total_revenue_usd"))

df_kpi_total_revenue.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("wholesale_catalog.gold.kpi_total_revenue")

display(df_kpi_total_revenue)

# ============================================================
# KPI 2: Total Quantity Sold
# ============================================================
df_kpi_qty = spark.table("wholesale_catalog.gold.fact_sales") \
    .filter(col("invoice_status") != "CANCELLED") \
    .agg(spark_sum("quantity").alias("total_quantity_sold"))

df_kpi_qty.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("wholesale_catalog.gold.kpi_total_quantity")

display(df_kpi_qty)

# ============================================================
# KPI 3: Average Invoice Value
# ============================================================
df_invoice_totals = spark.table("wholesale_catalog.gold.fact_sales") \
    .filter(col("invoice_status") != "CANCELLED") \
    .groupBy("invoice_id") \
    .agg(spark_sum("line_total_usd").alias("invoice_total_usd"))

df_kpi_avg_invoice = df_invoice_totals \
    .agg(spark_round(avg("invoice_total_usd"), 2).alias("avg_invoice_value_usd"))

df_kpi_avg_invoice.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("wholesale_catalog.gold.kpi_avg_invoice_value")

display(df_kpi_avg_invoice)

# ============================================================
# KPI 4: Top Customers by Revenue
# ============================================================
df_customers = spark.table("wholesale_catalog.gold.dim_customers")
df_sales     = spark.table("wholesale_catalog.gold.fact_sales") \
                    .filter(col("invoice_status") != "CANCELLED")

df_kpi_top_customers = df_sales \
    .groupBy("customer_id") \
    .agg(spark_round(spark_sum("line_total_usd"), 2).alias("total_revenue_usd")) \
    .join(df_customers.select("customer_id", "customer_name", "region"), "customer_id") \
    .orderBy(col("total_revenue_usd").desc()) \
    .limit(20)

df_kpi_top_customers.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("wholesale_catalog.gold.kpi_top_customers")

display(df_kpi_top_customers)

# ============================================================
# KPI 5: Top Products by Revenue
# ============================================================
df_products = spark.table("wholesale_catalog.gold.dim_products")

df_kpi_top_products = df_sales \
    .groupBy("product_id") \
    .agg(
        spark_round(spark_sum("line_total_usd"), 2).alias("total_revenue_usd"),
        spark_sum("quantity").alias("total_qty_sold")
    ) \
    .join(df_products.select("product_id", "product_name", "category"), "product_id") \
    .orderBy(col("total_revenue_usd").desc()) \
    .limit(20)

df_kpi_top_products.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("wholesale_catalog.gold.kpi_top_products")

display(df_kpi_top_products)

# ============================================================
# KPI 6: Invoice Cancellation Rate
# ============================================================
df_all_invoices = spark.table("wholesale_catalog.silver.invoices")

df_kpi_cancellation = df_all_invoices.agg(
    count("invoice_id").alias("total_invoices"),
    spark_sum(when(col("invoice_status") == "CANCELLED", 1).otherwise(0)).alias("cancelled_invoices")
).withColumn(
    "cancellation_rate_pct",
    spark_round(col("cancelled_invoices") / col("total_invoices") * 100, 2)
)

df_kpi_cancellation.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("wholesale_catalog.gold.kpi_cancellation_rate")

display(df_kpi_cancellation)

# ============================================================
# KPI 7: Discount Impact per Customer
# ============================================================
df_kpi_discount = df_sales \
    .groupBy("customer_id") \
    .agg(
        spark_round(avg("discount_pct"), 2).alias("avg_discount_pct"),
        spark_round(spark_sum(
            col("quantity") * col("unit_price") * (col("discount_pct") / 100)
        ), 2).alias("total_discount_given_usd")
    ) \
    .join(df_customers.select("customer_id", "customer_name"), "customer_id") \
    .orderBy(col("total_discount_given_usd").desc())

df_kpi_discount.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("wholesale_catalog.gold.kpi_discount_impact")

display(df_kpi_discount)

# ============================================================
# KPI 8: Revenue by Region
# ============================================================
df_kpi_region = df_sales \
    .groupBy("region") \
    .agg(spark_round(spark_sum("line_total_usd"), 2).alias("total_revenue_usd")) \
    .orderBy(col("total_revenue_usd").desc())

df_kpi_region.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("wholesale_catalog.gold.kpi_revenue_by_region")

display(df_kpi_region)

# ============================================================
# KPI 9: Number of Invoices per Customer
# ============================================================
df_kpi_invoice_freq = df_all_invoices \
    .groupBy("customer_id") \
    .agg(countDistinct("invoice_id").alias("invoice_count")) \
    .join(df_customers.select("customer_id", "customer_name"), "customer_id") \
    .orderBy(col("invoice_count").desc())

df_kpi_invoice_freq.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("wholesale_catalog.gold.kpi_invoices_per_customer")

display(df_kpi_invoice_freq)
print("\nAll KPIs written to Gold layer!")
